# v27 self-prefix — 自分のPS前ラテラルGRを疑似typewellにして PF尤度に混ぜる(別ルート)

## 主催者ヒント(Igor Kuvaev)
- ラテラルの **PS点より前(既知prefix)の GR は typewell より高分解能**。
- **eval が TVT 上向き(prefix帯に戻る)区間**では、その坑井自身の prefix GR との相関が最良。
- ＝**地層を上向きに切る坑井の推論では、typewellより自分から得たデータの方が高分解能**。

## この実験(近傍版と同じやり方, ただし参照=自分のprefix)
- **self pseudo-typewell** = 自分の (GR, TVT_input) 既知prefix を TVTビン median で GR(TVT)化。
- PF尤度gridに `SELF_MIX_W` で混合(covered帯=prefixのTVT帯だけ置換, 帯外=typewell)。近傍版と同一機構。
- **typewellのみ vs self-prefix混合** を OOF(train)で比較(overall + 出力ブレンドa掃引)。近傍不要=**全坑井対象**・test安全(自分のprefixのみ使用, eval真値不使用)。
- 方向(上向き/下向き)の明示判定は**不要**: self-prefix は prefix の TVT帯だけ置換・帯外は typewell なので、gridが自動で処理(判定してもしなくても同じ)。

## 注意
- リーク無し: 使うのは既知prefix(TVT_input)とGRのみ。eval真値TVTは採点のみ。
- PF = v25 r0 torch/GPU(exp_xyknn と同一エンジン, N_SEED/lag等 cell engine の定数)。


In [ ]:
# ===== import + データパス自動判定 + train坑井一覧 =====
import os, glob, json, time, hashlib, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore"); np.seterr(all="ignore")

CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TRAIN = COMP / "train"
ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v27\artifacts_v27"); ART.mkdir(parents=True, exist_ok=True)

WELLS = sorted(os.path.basename(p).split("__")[0]
               for p in glob.glob(str(TRAIN / "*__horizontal_well.csv")))
print("COMP:", COMP, "| train wells:", len(WELLS), "| ART:", ART)

In [ ]:
# ===== typewell ロード(build_ctx用) + 対象=全train坑井 =====
SELF_MIX_W = 1.0     # 自分prefix疑似typewellの混合比(今の方法と同じ=1.0)
N_WELLS = None       # 検証を軽くする時に坑井数制限(例50)。None=全部
TW_CACHE = {}; _t0 = time.time()
for _k, _w in enumerate(WELLS):
    TW_CACHE[_w] = pd.read_csv(TRAIN / f"{_w}__typewell.csv")
    if (_k + 1) % 200 == 0: print(f"  typewell {_k+1}/{len(WELLS)} ({time.time()-_t0:.0f}s)")
sp_wells = list(WELLS)[:N_WELLS] if N_WELLS else list(WELLS)
print("対象train坑井:", len(sp_wells))

In [ ]:
# ===== データアクセサ + 整合性チェック =====
# valid坑井: test模擬(GR全区間 + TVT_input既知 + eval区間の隠し真値TVT)
# 参照坑井 : train全既知模擬(GR + TVT全真値)
_HW_CACHE = {}
def _load_hw(wid):
    if wid not in _HW_CACHE:
        _HW_CACHE[wid] = pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv")
    return _HW_CACHE[wid]

def ref_arrays(wid):
    hw = _load_hw(wid)
    return dict(wid=wid, md=hw["MD"].to_numpy(float), x=hw["X"].to_numpy(float),
                y=hw["Y"].to_numpy(float), z=hw["Z"].to_numpy(float),
                gr=hw["GR"].to_numpy(float), tvt=hw["TVT"].to_numpy(float))   # 全真値

def valid_arrays(wid):
    hw = _load_hw(wid)
    ev = hw["TVT_input"].isna().to_numpy()
    last_kn = hw.loc[hw["TVT_input"].notna(), "TVT_input"]
    return dict(wid=wid, md=hw["MD"].to_numpy(float), x=hw["X"].to_numpy(float),
                y=hw["Y"].to_numpy(float), z=hw["Z"].to_numpy(float),
                gr=hw["GR"].to_numpy(float),
                tvt_input=hw["TVT_input"].to_numpy(float),   # 既知prefix(eval=NaN)
                tvt_true=hw["TVT"].to_numpy(float),          # 採点用の隠し真値
                eval_mask=ev,
                last_known_tvt=float(last_kn.iloc[-1]) if len(last_kn) else np.nan,
                ids=[f"{wid}_{i}" for i in range(len(hw))])

In [ ]:
# ===== PF エンジン(v25 r0 torch/GPU 忠実移植) + 尤度grid(typewell / +疑似typewell) =====
# v25 submit_smoother の r0 torch エンジンをそのまま持込み(numpy移植でなくGPUバッチ=v25と等価)。
# grid(gg) を typewell / 混合(疑似typewell) で差し替えて A/B。出力は固定ラグ平滑(SMOOTH_LAG)。
import torch, time
V27_DEV = "cuda" if torch.cuda.is_available() else "cpu"; V27_DT = torch.float32
PF_N       = 400
N_SEED     = 32
SMOOTH_LAG = 192     # v25 6.786 と同じ固定ラグ。0=forward PF
MIX_W      = 1.0     # 疑似typewell混合比(covered帯; 1.0=実HWテンプレで完全置換=勝者, 0=typewellのみ)
POOL_MODE  = "median" # 複数参照の疑似typewell生成: "mean" / "median"(平均のボケ回避) / "best1"(最類似1本)
MIX_MAX_REFS = 1      # 混合を適用する参照本数の上限(1=ペアのみ。参照多数群はtypewellが強いので混ぜない=ゲート)
SEED       = 0
CHUNK      = 16      # GPUバッチ坑井数(OOMなら下げる)
print("v27 PF device:", V27_DEV, "| torch:", torch.__version__)

R0 = {
    "n_particles": 400, "likelihood_scale": 10.536319667317654,
    "init_pos_std": 0.9331541486020234, "init_rate_std": 0.00047753793163756304,
    "mom": 0.9997911117425017, "vn": 0.0006297672017716754, "pn": 0.005014009140727263,
    "rp": 0.7627309890952053, "rr": 0.00045663430474523125, "resamp": 0.032049849143995945,
    "gr_sig_min": 11.865793233968713, "gr_sig_max": 44.98927964612316, "gr_sig_def": 119.25397464636981,
    "gr_sig_mult": 3.058318309943017, "tvt_clip_margin": 150.56111180234504, "rate_clip": 2.0,
    "gr_power": 1.7826896851395453, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0,
    "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 0.013666550145707291, "use_phys": False,
}
P0 = {**R0, "n_particles": PF_N, "smooth_lag": SMOOTH_LAG}

def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def _pf_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=V27_DT):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w*(pos-zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i-L] = (w*old).sum(1) - z[:, i-L]
            buf[:, ptr, :] = pos; ptr = (ptr+1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T-L), T):
            pts_s[:, j] = (w*buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik

def _pad(inps, device, dtype=V27_DT):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pf, ps, ll = _pf_smoother_core(
        rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
        pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
        pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
        P, N, device, gen)
    pf = pf.view(W, S, -1).double().cpu().numpy(); ps = ps.view(W, S, -1).double().cpu().numpy()
    ll = ll.view(W, S).cpu().numpy()
    out_f = []; out_s = []
    for w in range(W):
        T = len(inps[w]["md"]); llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw-mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww/s if s > 1e-300 else np.full(S, 1.0/S)
        else:
            ww = np.full(S, 1.0/S)
        out_f.append((ww[:, None]*pf[w, :, :T]).sum(0))
        out_s.append((ww[:, None]*ps[w, :, :T]).sum(0))
    if device == "cuda": torch.cuda.synchronize()
    return out_f, out_s

# --- v27: valid坑井の PF入力 + 尤度grid(typewell / 混合) ---
def build_ctx(va, tw, P=P0):
    md = va["md"]; z = va["z"]; ev = va["eval_mask"]; kn = ~ev
    tw = tw.sort_values("TVT"); tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    tw_gr_pf = _smooth(tw_gr, fb, P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(tw_tvt, tw_gr_pf)
    gr_sm = _smooth(va["gr"], fb, P["hgr_smooth_r"])
    ki = np.where(kn)[0]
    if len(ki) >= 20:
        resid = gr_sm[ki] - np.interp(va["tvt_input"][ki], tw_tvt, tw_gr_pf); gs = float(np.nanstd(resid))
        if not np.isfinite(gs) or gs <= 0: gs = P["gr_sig_def"]
    else:
        gs = P["gr_sig_def"]
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    ls = float(va["tvt_input"][ki[-1]] + z[ki[-1]])
    tl = ki[-30:]; dt = np.diff(va["tvt_input"][tl]); dz = np.diff(z[tl]); dm = np.diff(md[tl]); mm = dm > 0
    ir = float(np.median((dt + dz)[mm] / dm[mm])) if int(mm.sum()) >= 3 else 0.0
    return dict(md=md[ev].astype(np.float64), z=z[ev].astype(np.float64), gr=gr_sm[ev].astype(np.float64),
                gs=gs, ls=ls, ir=ir, gmin=float(gmin), gst=float(gst), gg_tw=np.asarray(gg, np.float64),
                kn_tvt=va["tvt_input"][ki].astype(np.float64), kn_gr=gr_sm[ki].astype(np.float64))  # best1用

def to_inp(ctx, gg, N):
    return dict(md=ctx["md"], z=ctx["z"], gr=ctx["gr"], gg=np.asarray(gg, np.float64),
                gmin=ctx["gmin"], gst=ctx["gst"], gs=ctx["gs"], ls=ctx["ls"], ir=ctx["ir"], N=N)

def _ref_pseudo(r, gmin, step, G):   # 参照1本を疑似typewell(TVTビン平均GR)に
    rt = r["tvt"]; rg = _smooth(r["gr"], float(np.nanmean(r["gr"])), P0["hgr_smooth_r"]).astype(float)
    m = np.isfinite(rt) & np.isfinite(rg); ii = np.round((rt[m] - gmin) / step).astype(int)
    ok = (ii >= 0) & (ii < G); ii, v = ii[ok], rg[m][ok]
    s = np.bincount(ii, weights=v, minlength=G); c = np.bincount(ii, minlength=G)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

def mixed_grid(ctx, refs, w_hw=MIX_W, pool=POOL_MODE):
    gg = ctx["gg_tw"].copy(); gmin = ctx["gmin"]; step = ctx["gst"]; G = len(gg)
    # best1: valid の既知prefix GR を最も再現する参照1本を選ぶ(spatialでなくGR-fit)
    if pool == "best1" and len(refs) > 1:
        kb = np.round((ctx["kn_tvt"] - gmin) / step).astype(int); vk = (kb >= 0) & (kb < G)
        best, bres = refs[0], np.inf
        for r in refs:
            gh = _ref_pseudo(r, gmin, step, G); pred = gh[kb[vk]]
            res = ctx["kn_gr"][vk] - pred; res = res[np.isfinite(res)]
            rr = float(np.sqrt(np.mean(res * res))) if len(res) else np.inf
            if rr < bres: bres, best = rr, r
        refs = [best]
    rt = np.concatenate([r["tvt"] for r in refs])
    rg = np.concatenate([_smooth(r["gr"], float(np.nanmean(r["gr"])), P0["hgr_smooth_r"]).astype(float) for r in refs])
    m = np.isfinite(rt) & np.isfinite(rg); rt, rg = rt[m], rg[m]
    idx = np.round((rt - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, rgv = idx[ok], rg[ok]
    if pool == "median":
        ghw = np.full(G, np.nan); med = pd.Series(rgv).groupby(idx).median()
        ghw[med.index.to_numpy()] = med.to_numpy()
    else:  # mean / best1(単一参照)
        sums = np.bincount(idx, weights=rgv, minlength=G); cnts = np.bincount(idx, minlength=G)
        ghw = np.where(cnts > 0, sums / np.maximum(cnts, 1), np.nan)
    have = np.isfinite(ghw); gm = gg.copy(); gm[have] = w_hw * ghw[have] + (1 - w_hw) * gg[have]
    return gm

def run_batched(inps, P=P0, chunk=CHUNK):
    outs = [None] * len(inps)
    for c0 in range(0, len(inps), chunk):
        _, os_ = run_smoother_ens(inps[c0:c0 + chunk], P, V27_DEV, SEED, N_SEED)
        for j, o in enumerate(os_): outs[c0 + j] = o
        if V27_DEV == "cuda": torch.cuda.empty_cache()
    return outs
print("PF engine (v25 r0 torch/GPU) ready | N=%d n_seed=%d lag=%d mix_w=%.2f" % (PF_N, N_SEED, SMOOTH_LAG, MIX_W))

In [ ]:
# ===== 自分の既知prefix → 疑似typewell(近傍版と同じ TVTビン median) → grid に混合 =====
def build_grid_self(ctx, w_hw=SELF_MIX_W):
    gg = ctx["gg_tw"].copy(); gmin = ctx["gmin"]; step = ctx["gst"]; G = len(gg)
    kt = np.asarray(ctx["kn_tvt"], float); kg = np.asarray(ctx["kn_gr"], float)  # 自分prefix(TVT_input, 平滑GR)
    m = np.isfinite(kt) & np.isfinite(kg); kt, kg = kt[m], kg[m]
    idx = np.round((kt - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, v = idx[ok], kg[ok]
    ghw = np.full(G, np.nan)
    if len(v):
        med = pd.Series(v).groupby(idx).median(); ghw[med.index.to_numpy()] = med.to_numpy()
    have = np.isfinite(ghw); gg[have] = w_hw * ghw[have] + (1 - w_hw) * gg[have]
    return gg
print("build_grid_self ready | SELF_MIX_W=%.2f (自分prefixのGR-TVTを混ぜる)" % SELF_MIX_W)

In [ ]:
# ===== A/B: typewellのみ vs 自分prefix混合 (OOF 全train坑井) =====
va_list = []; ctxs = []; used = []
for w in sp_wells:
    try:
        va = valid_arrays(w)
        if int(va["eval_mask"].sum()) == 0 or int((~va["eval_mask"]).sum()) < 20: continue
        ctx = build_ctx(va, TW_CACHE[w], P0)
    except Exception:
        continue
    va_list.append(va); ctxs.append(ctx); used.append(w)
print("使用坑井:", len(used))
true_list = [va["tvt_true"][va["eval_mask"]] for va in va_list]
inps_tw = [to_inp(ctxs[i], ctxs[i]["gg_tw"], PF_N) for i in range(len(used))]
inps_sf = [to_inp(ctxs[i], build_grid_self(ctxs[i]), PF_N) for i in range(len(used))]
_t0 = time.time()
out_tw = run_batched(inps_tw, P0); out_sf = run_batched(inps_sf, P0)
print(f"PF done {time.time()-_t0:.0f}s on {V27_DEV} | wells={len(used)} n_seed={N_SEED} lag={SMOOTH_LAG}")

_ALL = list(range(len(used)))
TW = [np.asarray(out_tw[i], float)[:len(true_list[i])] for i in _ALL]
SF = [np.asarray(out_sf[i], float)[:len(true_list[i])] for i in _ALL]
def _rmse(pl):
    E = np.concatenate([pl[i] - true_list[i] for i in _ALL]); E = E[np.isfinite(E)]
    return float(np.sqrt(np.mean(E*E))) if len(E) else float("nan")
print("=== overall RMSE (全 %d坑井プール) ===" % len(used))
print("  hold        : %.4f" % _rmse([np.full(len(true_list[i]), va_list[i]["last_known_tvt"]) for i in _ALL]))
print("  PF typewell : %.4f" % _rmse(TW))
print("  PF +self    : %.4f" % _rmse(SF))
print("\n=== 出力ブレンド a: (1-a)*tw + a*self (PF再実行なし) ===")
for a in [0.0, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
    print("  a=%.1f: %.4f" % (a, _rmse([(1-a)*TW[i] + a*SF[i] for i in _ALL])))
print("\n※ self-prefix=自分のPS前GR(TVT_input)を PF尤度に混合(近傍版と同じ機構)。方向判定は不要(gridが自動)。")

In [ ]:
# ===== self-prefix: 出力ブレンド探索 + 上向き度別 (PF再実行なし。前セルの TW/SF/true_list/ctxs を使用) =====
def _rmse_pool(pl, idxs):
    E = np.concatenate([pl[i] - true_list[i] for i in idxs]); E = E[np.isfinite(E)]
    return float(np.sqrt(np.mean(E * E))) if len(E) else float("nan")
def _blend_pool(a, idxs):
    return _rmse_pool([(1 - a) * TW[i] + a * SF[i] for i in _ALL], idxs)

_res = [(round(0.05 * k, 2), _blend_pool(round(0.05 * k, 2), _ALL)) for k in range(21)]
_best = min(_res, key=lambda x: x[1])
print("=== 出力ブレンド a: (1-a)*tw + a*self [全 %d坑井] ===" % len(_ALL))
print("  " + "  ".join("a%.1f=%.3f" % (a, o) for a, o in _res if int(round(a * 100)) % 10 == 0))
print("  ★best a=%.2f (%.4f) / tw(a0)=%.4f" % (_best[0], _best[1], _res[0][1]))

def _prng(ctx):
    kt = np.asarray(ctx["kn_tvt"], float); kt = kt[np.isfinite(kt)]
    return (float(kt.min()), float(kt.max())) if len(kt) else (np.inf, -np.inf)
_pr = [_prng(c) for c in ctxs]
_frac = [float(((true_list[i] >= _pr[i][0]) & (true_list[i] <= _pr[i][1])).mean()) for i in _ALL]
print("\n=== 上向き度別(eval中 prefix帯割合): tw vs self(a=1) vs best-a=%.2f ===" % _best[0])
for lo, hi in [(0.0, 0.2), (0.2, 0.5), (0.5, 0.8), (0.8, 1.01)]:
    idxs = [i for i in _ALL if lo <= _frac[i] < hi]
    if not idxs: continue
    otw = _rmse_pool(TW, idxs); osf = _rmse_pool(SF, idxs); ob = _blend_pool(_best[0], idxs)
    print("  上向き度[%.1f-%.1f] n=%4d: tw %.3f  self %.3f  best-a %.3f  (%+.3f)" % (lo, hi, len(idxs), otw, osf, ob, otw - ob))